In [1]:
import pandas as pd
import duckdb

In [2]:
df_uber = pd.read_parquet('Drafts/df_uber_parquet.parquet')
df_pick_up = pd.read_parquet('Final/Dim_Date_Pick_Up.parquet')
df_drop_off = pd.read_parquet('Final/Dim_Date_Drop_Off.parquet')

In [3]:
df_uber.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'pickup_longitude',
       'pickup_latitude', 'RatecodeID', 'store_and_fwd_flag',
       'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount',
       'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
       'improvement_surcharge', 'total_amount'],
      dtype='object')

In [4]:
df_pick_up.columns

Index(['pick_up_time_id', 'date_time', 'day', 'hour', 'month', 'year',
       'weekday', 'is_weekend'],
      dtype='object')

In [5]:
df_drop_off.columns

Index(['drop_off_time_id', 'date_time', 'day', 'hour', 'month', 'year',
       'weekday', 'is_weekend'],
      dtype='object')

In [6]:
Dim_Travel = duckdb.sql(
    """
        SELECT p.pick_up_time_id AS pick_up_id, d.drop_off_time_id AS drop_off_id
        FROM df_uber u
        LEFT JOIN df_pick_up p
        ON u.tpep_pickup_datetime = p.date_time
        LEFT JOIN df_drop_off d
        ON u.tpep_dropoff_datetime = d.date_time
    """
).df()

In [7]:
Dim_Travel.head()

,pick_up_id,drop_off_id
0,1,1
1,1,2
2,1,3
3,1,4
4,1,4


In [8]:
Dim_Travel.insert(0,'travel_id', range(1,len(Dim_Travel)+1))

In [9]:
Dim_Travel.head()

,travel_id,pick_up_id,drop_off_id
0,1,1,1
1,2,1,2
2,3,1,3
3,4,1,4
4,5,1,4


In [10]:
Dim_Travel.to_parquet('Final/Dim_Travel.parquet',index = False)